# CPE 310 — Group 7: P2P File Transfer & Integrity Verification System
**Federal University Oye-Ekiti (FUOYE)**

This notebook demonstrates the full P2P system:
- **Part 1** — Run the automated test suite (45 tests) with visible output
- **Part 2** — Live simulation showing the system working in a real-life scenario


## Setup — Clone Repository & Install Dependencies

In [ ]:
# Clone the project from GitHub
!git clone https://github.com/samuelomoyeni09-pixel/p2p_file_transfer.git
%cd p2p_file_transfer
!pip install -r requirements.txt -q

---
## PART 1 — Automated Test Suite (45 Tests)
Each test prints what input was given and how the system responded.

> The `-s` flag disables pytest's stdout capture so all `print()` output is visible.

In [ ]:
!python -m pytest tests/test_p2p.py -s -v

---
## PART 2 — Live Simulation Demo
This runs a full end-to-end simulation of the P2P network with multiple nodes, showing:
- File chunking and metadata
- Seed node sharing a file
- Tracker ranking peers by bandwidth
- Clean download (PushProtocol, 0% corruption)
- Noisy download with 40% simulated corruption and automatic retries
- Tampered chunk detection via SHA-256 verify()
- Swarm membership
- Rate limiter enforcement

In [ ]:
!python demo.py

---
## PART 2 (Inline) — Run Demo Interactively in This Notebook
You can also run the demo code directly in the notebook cells below.

In [ ]:
import hashlib, random
from src.chunk_data import ChunkData
from src.exceptions import ChecksumMismatchError, PeerNotFoundError
from src.file_metadata import FileMetadata
from src.nodes import DataPeerNode, MetadataTrackerNode, SeedNode
from src.protocols import PullProtocol, PushProtocol
from src.transfer_session import TransferSession

SEP = '=' * 60
def bar(pct, width=30):
    filled = int(width * pct / 100)
    return '[' + '#' * filled + '-' * (width - filled) + f'] {pct:.1f}%'

print('All imports successful.')

In [ ]:
# ── SCENE 1: File Metadata ──────────────────────────────────
print(SEP)
print('  SCENE 1: File Creation & Metadata')
print(SEP)

FILE_CONTENT = (
    b'CPE 310 Group 7 - P2P File Transfer System\n'
    b'Federal University Oye-Ekiti (FUOYE)\n'
    b'Each chunk is verified with SHA-256 to detect corruption.\n'
) * 10

CHUNK_SIZE = 256
metadata = FileMetadata('lecture_notes.txt', FILE_CONTENT, chunk_size_bytes=CHUNK_SIZE)

print(f'  Filename   : {metadata.filename}')
print(f'  File size  : {metadata.size_bytes} bytes')
print(f'  Chunk size : {metadata.chunk_size_bytes} bytes')
print(f'  Chunks     : {metadata.chunk_count}')
print(f'  SHA-256    : {metadata.sha256_hash[:32]}...')

In [ ]:
# ── SCENE 2 & 3: Seed + Tracker ────────────────────────────
print(SEP)
print('  SCENE 2: Seed Node Shares File')
print(SEP)

seed = SeedNode('seed-01', '192.168.1.1:9000', bandwidth=100.0, max_bandwidth_bps=500_000)
seed.share_file(FILE_CONTENT, 'lecture_notes.txt', chunk_size=CHUNK_SIZE)
print(f'  Seed: {seed.node_id} [{seed.address}]  bandwidth={seed.bandwidth} Mbps')
print(f'  Has file? {seed.has_file(metadata.sha256_hash)}')

print(f'\n{SEP}')
print('  SCENE 3: Tracker Registers Peers')
print(SEP)

tracker = MetadataTrackerNode('tracker-01', '192.168.1.0:6969')
for pid, addr, bw in [('peer-A','192.168.1.10:8000',80.0),
                       ('peer-B','192.168.1.11:8001',45.0),
                       ('peer-C','192.168.1.12:8002',30.0)]:
    p = DataPeerNode(pid, addr, bandwidth=bw)
    p.share_file(FILE_CONTENT, 'lecture_notes.txt', chunk_size=CHUNK_SIZE)
    tracker.register_file(metadata, p)
    print(f'  Registered: {pid} [{addr}]  {bw} Mbps')

tracker.register_file(metadata, seed)
print(f'  Registered: {seed.node_id} [{seed.address}]  {seed.bandwidth} Mbps')

ranked = tracker.find_peers(metadata.sha256_hash)
print(f'\n  Peer ranking (fastest first):')
for i, p in enumerate(ranked, 1):
    print(f'    {i}. {p.node_id}  ({p.bandwidth} Mbps)')

In [ ]:
# ── SCENE 4: Clean PushProtocol Download ───────────────────
print(SEP)
print('  SCENE 4: Clean Download — PushProtocol (0% corruption)')
print(SEP)

downloader = DataPeerNode('peer-NEW', '192.168.1.99:8080', bandwidth=60.0)
best_peer  = tracker.find_peers(metadata.sha256_hash)[0]
session    = TransferSession(downloader, best_peer, metadata)
proto      = PushProtocol(corruption_probability=0.0)

print(f'  Downloader : {downloader.node_id}')
print(f'  Provider   : {best_peer.node_id}')
print(f'  Protocol   : {proto}\n')

proto.initiate(session)
for idx in range(metadata.chunk_count):
    chunk = proto.transfer_chunk(session, idx, provider=best_peer)
    session.mark_received(chunk)
    print(f'  Chunk {idx+1:>2}/{metadata.chunk_count}  size={len(chunk.data):>3}B  '
          f'checksum={chunk.checksum[:12]}...  verify={chunk.verify()}  '
          f'{bar(session.progress_pct)}')

print(f'\n  Complete: {session.is_complete()}')
print(f'  Whole-file SHA-256 verified: {proto.finalise(session)}')
print(f'  Retries: {session.retry_count}')

In [ ]:
# ── SCENE 5: Noisy PullProtocol with Retries ───────────────
print(SEP)
print('  SCENE 5: Noisy Network — PullProtocol (40% corruption + retries)')
print(SEP)

random.seed(7)
downloader2 = DataPeerNode('peer-STUDENT', '192.168.1.50:9090', bandwidth=55.0)
providers   = tracker.find_peers(metadata.sha256_hash)
session2    = TransferSession(downloader2, providers[0], metadata)
proto2      = PullProtocol(corruption_probability=0.4)

print(f'  Downloader : {downloader2.node_id}')
print(f'  Providers  : {[p.node_id for p in providers]}')
print(f'  Protocol   : {proto2}\n')

proto2.initiate(session2)
total_retries = 0

for chunk_idx in range(metadata.chunk_count):
    attempt = 0
    for provider in providers * 50:
        try:
            chunk = proto2.transfer_chunk(session2, chunk_idx, provider=provider)
            session2.mark_received(chunk)
            status = 'OK' if attempt == 0 else f'OK (after {attempt} retry)'
            print(f'  Chunk {chunk_idx+1:>2}/{metadata.chunk_count}  [{provider.node_id}]  '
                  f'{status:<22}  {bar(session2.progress_pct)}')
            break
        except ChecksumMismatchError as e:
            total_retries += 1
            attempt += 1
            session2.increment_retry()
            print(f'  Chunk {chunk_idx+1:>2}/{metadata.chunk_count}  [{provider.node_id}]  '
                  f'CORRUPTED! expected={e.expected[:8]}... got={e.received[:8]}...  Retrying...')

print(f'\n  Complete: {session2.is_complete()}')
print(f'  Whole-file SHA-256 verified: {proto2.finalise(session2)}')
print(f'  Total retries: {total_retries}')

In [ ]:
# ── SCENE 6: Tamper Detection ──────────────────────────────
print(SEP)
print('  SCENE 6: Integrity Verification — Tampered Chunk Detected')
print(SEP)

original = ChunkData(metadata.sha256_hash, 0, FILE_CONTENT[:CHUNK_SIZE])
tampered = ChunkData._from_raw(metadata.sha256_hash, 0, b'\x00' * CHUNK_SIZE, original.checksum)

print(f'  Original chunk[0] verify() = {original.verify()}  (data intact)')
print(f'  Tampered chunk[0] verify() = {tampered.verify()}  (data modified — mismatch!)')
print(f'  Expected: {original.checksum[:32]}...')
print(f'  Actual  : {hashlib.sha256(tampered.data).hexdigest()[:32]}...')

# ── SCENE 7: Swarm ─────────────────────────────────────────
print(f'\n{SEP}')
print('  SCENE 7: Swarm Membership')
print(SEP)

swarm = tracker.get_swarm(metadata.sha256_hash)
print(f'  Swarm size: {len(swarm)} peers')
for p in swarm:
    print(f'    - {p.node_id}  [{p.address}]  {p.bandwidth} Mbps')
print(f"\n  'peer-A' in swarm? {'peer-A' in swarm}")
print(f"  'ghost'  in swarm? {'ghost'  in swarm}")

# ── SCENE 8: Rate Limiter ──────────────────────────────────
print(f'\n{SEP}')
print('  SCENE 8: Rate Limiter Simulation')
print(SEP)

lseed = SeedNode('seed-limited', '10.0.0.1:9000', max_bandwidth_bps=500)
lseed.share_file(b'X' * 1000, 'bigfile.bin', chunk_size=100)
lmeta = FileMetadata('bigfile.bin', b'X' * 1000, 100)
print(f'  Upload cap: {lseed.max_bandwidth_bps} B/s\n')
for i in range(5):
    chunk = lseed.request_chunk(lmeta.sha256_hash, i)
    can   = lseed.can_send(len(chunk.data))
    print(f'  Chunk[{i}] size={len(chunk.data)}B  used={lseed.bytes_sent_this_second}B  can_send_next={can}')
    if not can:
        print('  >>> Rate limit reached! Simulating next second...')
        lseed.reset_bandwidth()

print(f'\n{SEP}')
print('  SIMULATION COMPLETE — All scenarios demonstrated successfully.')
print(SEP)